Convert PLSCADD Structure Usage Report to XLS  
Before start please save the report as StrUsageReport.xml (right click - save all as XML)  

Start with importing libraries

In [1]:
# @title
import xml.etree.ElementTree as ET
import openpyxl
from pathlib import Path

Paths

In [2]:
workdir = Path(r'C:\Users\Igor.Bertyaev.APD\OneDrive - APD\_IGOR\__NZ\Z_APD01039_OTA-WKM-C Earthwire Replacement\reports\structure usage')  # working directory
xmlfile = workdir / 'exist_26-06.xml'  # input XML file
xlsfile = workdir / 'ExistEW_StrUsage.xlsx'  # output XLSX file

Read XML file and determine funtions

In [3]:
tree = ET.parse(xmlfile)
root = tree.getroot()

In [4]:
# @title
def tabs(xml_element, sheet):
    # parse each tab

    # start with headers
    for head in range(len(xml_element[0])):
        sheet.cell(row=1, column=head+1, value=xml_element[0][head].tag)

    # Excel max rows limit
    max_rows = 1048576
    rows_to_process = min(len(xml_element), max_rows - 1)  # -1 for header row
    
    # Iterate through the columns and rows within the specified range
    for row_idx in range(2, rows_to_process+2):
        # start from 2 because first is header
        sheet.cell(row=row_idx, column=1, value=row_idx-1)
        for col_idx in range(2, len(xml_element[row_idx-2])+2):
            # +2 because excel starts from 1 and first is empty in xml
            sheet.cell(row=row_idx, column=col_idx-1, value=xml_element[row_idx-2][col_idx-2].text)
    
    # Warn if data was truncated
    if len(xml_element) > rows_to_process:
        print(f"Warning: Sheet '{sheet.title}' had {len(xml_element)} rows, but only {rows_to_process} rows were written (Excel limit: {max_rows})")


In [5]:
# @title
def excel_export(xml):
    # create excel for first tab
    # parse first tab

    tabs_names = [] # list of tables
    for n in xml[1:]:
        tabs_names.append(n.attrib['plsname'])
    tabs_names

    wb = openpyxl.Workbook()
    sheet = wb.active
    sheet.title = "Index"

    ### fill the first sheet with info
    sheet['B2'] = 'User: '
    sheet['B3'] = 'Project: '
    sheet['B4'] = 'Date: '
    sheet['C2'] = xml[0].get("user")
    sheet['C3'] = xml[0].get("project")
    sheet['C4'] = xml[0].get("date")
    sheet['B6'] = 'Tabs Index: '

    for ce in range(len(tabs_names)):
        sheet[str('B' + str(int(7 + ce)))] = 'Tab_' + str(ce+1) # write sheet name
        sheet[str('C' + str(int(7 + ce)))] = tabs_names[ce] # write table name
        wb.create_sheet('Tab_' + str(ce+1)) # create new sheet

    # write to each sheet
    tabs_n = 1
    for ta in xml[1:]:
        sheet = wb['Tab_' + str(tabs_n)]
        tabs(ta, sheet)
        tabs_n+=1


    wb.save(xlsfile)


Start processing

In [6]:
# @title
# start processing
excel_export(root)

For any questions and concerns please contact Igor Bertyaev

ToDo list to improve:


1.   Style spreadsheets a bit
2.   Would be good to use it for all kind of reports
3.   Try to work with any name of report - finding its name automatically

